# Example: Use-cases of convolution
This notebook aims to provide a series of use-cases, with varying degree of complexity and some of them making use of more advanced functionality of the framework.

## Convolving BinCodex Data
The main purpose of this code is to convolve population-synthesis results with star-formation rates. A recently developed new format for population-synthesis output is the BinCodex format ([Valli et al. 2023](https://ui.adsabs.harvard.edu/abs/2023arXiv231103431V/abstract)). This section shows how to modify the BinCodex datafiles and convolve that data.

We will open an example BinCodex file(s), extract the data frame, store it in an input hdf5 file fit for convolution, configure the convolution and perform a simple convolution.

In [1]:
import os
import json
import copy
import numpy as np
import astropy.units as u

from syntheticstellarpopconvolve import convolve
from syntheticstellarpopconvolve import default_convolution_config

from syntheticstellarpopconvolve.general_functions import temp_dir

TMP_DIR = temp_dir("notebooks", "notebook_convolution_usecase_bincodex", clean_path=True)

import h5py
import pkg_resources
import pandas as pd

# create file
input_hdf5_filename = os.path.join(TMP_DIR, 'input_hdf5.h5')
input_hdf5_file = h5py.File(input_hdf5_filename, 'w')

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# add group for events
input_hdf5_file.create_group("input_data/events")

# close 
input_hdf5_file.close()

/tmp/ipykernel_649149/759737130.py:15: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [2]:
# load as RLOF event-data as pandas df
example_BinCodex_events_filename = pkg_resources.resource_filename(
    "syntheticstellarpopconvolve", 'example_data/example_BinCodex.h5'
)

example_BinCodex_T0_events = pd.read_hdf(
    example_BinCodex_events_filename,
    "T0",
)

example_BinCodex_T0_header = pd.read_hdf(
    example_BinCodex_events_filename,
    "header",
)

# Add metallicity as a column
example_BinCodex_T0_events['metallicity'] = float(example_BinCodex_T0_header.iloc[0]['Z'])

# Add normalized_yield as a column manually. This column should be the product of the probability of system a system and a conversion factor to make it per-unit-mass 
# NOTE: newer versions of BinCodex may have this column included (or at least based on probability)
example_BinCodex_T0_events['normalized_yield'] = 1

# store the data frame in the hdf5file
example_BinCodex_T0_events.to_hdf(
    input_hdf5_filename, key="input_data/events/BinCodex_T0"
)

In [3]:
print(example_BinCodex_T0_events)

   ID  UID  SID       time  event  semiMajor  eccentricity  type1     mass1  \
0   0    0    2      0.000   -1.0    4.96684           0.0  121.0  0.862931   
1   0    0    3    440.859    0.0    2.00180           0.0  121.0  0.862931   
2   0    0    7    445.006   52.0    0.00462           0.0  121.0  0.823353   
3   0    0    7    445.006   52.0    0.00000           0.0  121.0  0.000000   
4   0    0    7   2626.830   52.0    0.00000           0.0  121.0  0.000000   
5   0    0    7   2668.960   52.0    0.00000           0.0  121.0  0.000000   
6   0    0    7   2774.230   52.0    0.00000           0.0  121.0  0.000000   
7   0    0    7   2906.240   52.0    0.00000           0.0  121.0  0.000000   
8   0    0    7   2911.490   52.0    0.00000           0.0  121.0  0.000000   
9   0    0    7  14000.000   84.0    0.00000           0.0  121.0  0.000000   

    radius1    Teff1  massHeCore1  type2     mass2    radius2    Teff2  \
0  0.773353  3.69997          0.0  121.0  0.783703   0.7

In [4]:
input_hdf5_file = h5py.File(input_hdf5_filename, 'a')

# Write population config to file
input_hdf5_file.create_dataset(
    "config/population", data=json.dumps({})
)

################
input_hdf5_file.close()

In [7]:
output_hdf5_filename = os.path.join(TMP_DIR, 'output_hdf5.h5')

#
convolution_config = copy.copy(default_convolution_config)
convolution_config['input_filename'] = input_hdf5_filename
convolution_config['output_filename'] = output_hdf5_filename
convolution_config['tmp_dir'] = TMP_DIR
convolution_config['redshift_interpolator_data_output_filename'] = os.path.join(TMP_DIR, 'interpolator_dict.p')

###
# convolution instructions
convolution_config['convolution_instructions'] = [
    {
        'input_data_type': 'event',
        'input_data_name': 'BinCodex_T0',
        'output_data_name': 'BinCodex_T0',
        'ignore_metallicity': True,
        'data_column_dict': {
            # required
            'delay_time': 'time',
            'normalized_yield': 'normalized_yield',
            # # optional*
            # 'metallicity': 'metallicity',
        },
    },
]

# 
convolution_config['time_type'] = 'lookback_time'

convolution_config['convolution_lookback_time_bin_edges'] = np.arange(2, 4, 0.5) * u.Gyr

# construct the sfr-dict (NOTE: this uses absolute SFR, not metallicity dependent)
sfr_dict = {}
sfr_dict['lookback_time_bin_edges'] = (np.arange(0, 10, 1) * u.Gyr).to(u.yr)
sfr_dict['starformation_array'] = 0.25 * np.ones(sfr_dict['lookback_time_bin_edges'].shape[0]-1) * u.Msun/u.yr # example of a constant star-formation rate. this could be anything of course.

# store
convolution_config['SFR_info'] = sfr_dict

In [9]:
# convolve
convolve(config=convolution_config)

# Show some of the content
with h5py.File(convolution_config['output_filename'], 'r') as output_hdf5_file:
    print(output_hdf5_file['output_data/'].keys())
    print(output_hdf5_file['output_data/event/'].keys())
    print(output_hdf5_file['output_data/event/BinCodex_T0/'].keys())
    print(output_hdf5_file['output_data/event/BinCodex_T0/BinCodex_T0'].keys())
    print(output_hdf5_file['output_data/event/BinCodex_T0/BinCodex_T0/convolution_results'].keys())
    print(output_hdf5_file['output_data/event/BinCodex_T0/BinCodex_T0/convolution_results']['2.25 Gyr'][()])



<KeysViewHDF5 ['event']>
<KeysViewHDF5 ['BinCodex_T0']>
<KeysViewHDF5 ['BinCodex_T0']>
<KeysViewHDF5 ['convolution_results']>
<KeysViewHDF5 ['2.25 Gyr', '2.75 Gyr', '3.25 Gyr']>
[0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25]


## Convolving data with Lgalaxies merger-tree info


## Cosmological gravitational-wave merger and supernova transient rate density

The original project this codebase was designed for was to calculate the cosmological binary black-hole (BHBH) gravitational-wave merger and supernova transient rate density ([Hendriks et. al. 2023](https://doi.org/10.1093/mnras/stad2857)).
 
The following section will flesh out the steps to perform a similar calculation.

Several ingredients are necessary here:
- population-synthesis results that contain BHBH systems and several supernova types
- A cosmological starformation rate density

This convolution is an event-based backward convolution.

In [7]:
import numpy as np

from syntheticstellarpopconvolve.starformation_rate_distributions import starformation_rate_distribution_vanSon2023
from syntheticstellarpopconvolve.metallicity_distributions import metallicity_distribution_vanSon2022

############
# configure the convolution time-bins
convolution_stepsize = 0.001

convolution_time_bins = np.arange(
    1e-6 - 0.5 * convolution_stepsize,
    10
    + 0.5 * convolution_stepsize,
    convolution_stepsize,
)
convolution_time_centers = (
    convolution_time_bins[1:] + convolution_time_bins[:-1]
) / 2

###############
# Get sfr values
sfr_values = starformation_rate_distribution_vanSon2023(
    np.array(convolution_time_centers),
)

print(sfr_values)

# ###############
# # Calculate the metallicity distribution dP/dlogZ grid
# dPdlogZ, metallicities, _ = metallicity_distribution_vanSon2022(
#     np.array(convolution_configuration["time_centers"]),
#     0,
#     -8,
#     # metallicity distribution settings for vanSon21
#     **{
#         "mu0": 0.025,
#         "muz": -0.05,
#         "sigma_0": 1.125,
#         "sigma_z": 0.05,
#         "alpha": -1.77,
#     }
#     step_logZ=0.05,
# )

# # Calculate full probability
# P = dPdlogZ

# # Multiply by sfr:
# MSSFR = (sfr_values * P.T).T.value
# X, Y = np.meshgrid(convolution_configuration["time_centers"], metallicities)

[0.01999704 0.02002662 0.02005622 ... 0.00332537 0.00332404 0.00332271] solMass / (yr Mpc3)


In [ ]:
import matplotlib.pyplot as plt

##################
# Set up figure logic
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(nrows=1, ncols=11)

ax = fig.add_subplot(gs[:, :-2])
ax_cb = fig.add_subplot(gs[:, -1])

##################
# Plot the data

# Get the normalisation
if scale == "linear":
    norm = colors.Normalize(vmin=MSSFR.min(), vmax=MSSFR.max())
elif scale == "log":
    norm = colors.LogNorm(
        vmin=10 ** (np.log10(MSSFR.max()) - 3),
        vmax=MSSFR.max(),
    )

##################
# Plot the MSSFR results
_ = ax.pcolormesh(
    X,
    Y,
    MSSFR.T,
    norm=norm,
    shading="auto",
    antialiased=plot_settings.get("antialiased", True),
    rasterized=plot_settings.get("rasterized", True),
)

# make colorbar
cb = matplotlib.colorbar.ColorbarBase(
    ax_cb, norm=norm, extend="min" if scale == "log" else None
)

##################
# Plot the extent of metallicity bins
indicator_lines_linewidth = 2
ax.hlines(
    [metallicity_bins.min(), metallicity_bins.max()],
    color="red",
    lw=indicator_lines_linewidth,
    xmin=min(convolution_configuration["time_centers"]),
    xmax=max(convolution_configuration["time_centers"]),
)

##################
# Plot the included metallicities
ax.hlines(
    metallicity_bin_centers,
    color="orange",
    lw=indicator_lines_linewidth,
    linestyle="--",
    alpha=1,
    xmax=0.2,
    xmin=0,
)

# Make up
ax.set_yscale("log")
ax.set_ylabel(r"Metallicity [$Z$]")
ax.set_xlabel(r"Redshift [$z$]")
# ax.set_title("Metallicity specific star formation rate density", fontsize=28)
# cb.ax.set_ylabel(r"$\frac{d\ \mathrm{SFR}}{dZ}$ [$M_{\odot} yr^{-1} Gpc^{-3}$]")
cb.ax.set_ylabel(r"$\mathrm{SFR}(Z,z)$ [$M_{\odot}\,yr^{-1}\,Gpc^{-3}$]")


In [ ]:
##################
#

# create file
input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# add group for events
input_hdf5_file.create_group("input_data/events")

# Write population config to file
input_hdf5_file.create_dataset("config/population", data=json.dumps({}))

# close
input_hdf5_file.close()

# store the data frame in the hdf5file
wd_binaries.to_hdf(input_hdf5_filename, key="input_data/events/stochastic_example")

#
convolution_config = copy.copy(default_convolution_config)
convolution_config["input_filename"] = input_hdf5_filename
convolution_config["output_filename"] = output_hdf5_filename
convolution_config["tmp_dir"] = TMP_DIR
convolution_config["redshift_interpolator_data_output_filename"] = os.path.join(
    TMP_DIR, "interpolator_dict.p"
)
convolution_config["multiply_by_time_binsize"] = False

###
# convolution instructions
convolution_config["convolution_instructions"] = [
    {
        "input_data_type": "event",
        "convolution_type": "integration",
        "input_data_name": "BHBH_example",
        "output_data_name": "BHBH_example",
        "ignore_metallicity": False,
        "data_column_dict": {
            # required
            "normalized_yield": "normalized_yield",
            "delay_time": {"column_name": "time", "unit": u.Myr},
        },
    },
]

#
convolution_config["time_type"] = "redshift"

# configure the convolution time-bins
convolution_stepsize = 0.001
convolution_settings["stepsize"] = convolution_stepsize
convolution_settings["time_bins"] = np.arange(
    1e-6 - 0.5 * convolution_stepsize,
    (convolution_settings["max_interpolation_redshift"] - 1)
    + 0.5 * convolution_stepsize,
    convolution_stepsize,
)
convolution_settings["time_centers"] = (
    convolution_settings["time_bins"][1:] + convolution_settings["time_bins"][:-1]
) / 2

# configure the sfr-dict


# store
convolution_config["SFR_info"] = sfr_dict

input_hdf5_file = h5py.File(input_hdf5_filename, "r")

# convolve
convolve(config=convolution_config)


With the full datasets for .. availble on https://zenodo.org/records/8083112, and a better resolved convolution, we can recreate the following figures

## Surface density of binary objects in the SMC
related https://arxiv.org/abs/2406.01420

## LISA UCB convolution of double white-dwarf binaries at current-day (convolution-by-sampling)


This example fleshes out the steps required to estimate the population of observable double white dwarf systems in the LISA band. Relevant studies are: https://arxiv.org/abs/2405.20484





Several ingredients are necessary here:
- population-synthesis results that contain white-dwarfs
- a Milky-Way galaxy star formation rate history model
- a method to evolve 

Convolution-by-sampling was developed especially for this project, as we want to 'generate' double white dwarf systems at a certain lookback time, and evolve them (through gravitational-wave radiation) to the present day.

The convolution broadly is done as follows:
- In a given lookback-time bin we calculate the total mass formed into stars
- We use that to generate double white dwarf systems (using mass_formed * yield-per-mass-formed) 
- We assign a birth time to these systems (with values bound by the edges of the lookback time bin)
- We 'evolve' these systems up to the current day under the influence of gravitational-wave radiation. We make use of Legwork ([Wagg et al 2021](https://ui.adsabs.harvard.edu/abs/2022ApJS..260...52W/abstract)) in this example.
- Filter out certain systems (those that would interact, those that would merge, etc)
- Calculate detection probabilities for the rest based on their position (either randomly assigned or motivated by a spatially-defined SFH) in the Milkyway and their system properties.
- Use this information to predict observable populations of DWD systems

In the following piece of code I show how we do this.

In [ ]:
"""
Functions to convolve the T0 format with sampling

TODO: move the calculations to the post-convolution hook
TODO: determine which systems that are (at present day) in the lisa frequency range should have interacted through RLOF
TODO: of the systems that are not RLOFing and are within the lisa waveband, store: indices, source.f_orb_now. the rest can be retrieved elsewhere
"""

import os
import json
import time
import copy
import astropy.units as u
import legwork as lw
import numpy as np
import astropy.constants as const
import pandas as pd
import h5py

from syntheticstellarpopconvolve import convolve, default_convolution_config
from syntheticstellarpopconvolve.general_functions import temp_dir

from mass_normalisation import get_mass_norm
from functions import get_period, is_rlofing
from DrawPositionsSeparable import sample_distances_simple
from syntheticstellarpopconvolve.convolve_stochastically import (
    select_dict_entries_with_new_indices,
)

TMP_DIR = temp_dir("code", "convolve_stochastically", clean_path=True)


def post_convolution_function(
    config, job_dict, sfr_dict, data_dict, result_dict, convolution_instruction
):
    """
    Post-convolution function to handle integrating the systems forward in time and finding those that end up in the LISA waveband.

    using local_indices to select everything and using Alexey's distance sampler to handle sampling the distances
    """

    # unpack data
    system_indices = result_dict["indices"]
    event_lookback_times = result_dict["event_lookback_times"]
    local_indices = np.arange(len(system_indices))

    # select system properties
    sma = data_dict["semimajor_axis"][system_indices]
    m_1 = data_dict["mass1"][system_indices]
    m_2 = data_dict["mass2"][system_indices]
    eccentricity = data_dict["eccentricity"][system_indices]
    periods = get_period(sma, m_1, m_2)
    f_orb_i = (1 / periods).to(u.Hz)

    # sample distances
    dist = sample_distances_simple(NBin=len(system_indices), age=age)

    #########
    # Set up legwork sources
    sources = lw.source.Source(
        m_1=m_1,
        m_2=m_2,
        ecc=eccentricity,
        f_orb=f_orb_i,
        dist=dist,
        interpolate_g=len(local_indices) > 1000,
    )

    ##########
    # TODO: use the below steps to improve the selection of systems in band
    # ask; how long until separation = RLOF separatin
    # then: how long does it take until with current separation to go to that seperation
    # then compare that time to
    # check if the distance matches a frequency thats within the waveband.

    #########
    # Evolve the systems until today
    t_evol = event_lookback_times

    sources.evolve_sources(t_evol)

    f_orb_now = sources.f_orb

    ####
    # categorisations
    lower_bound_LISA_passband = 1e-5 * u.Hz
    upper_bound_LISA_passband = 1e-1 * u.Hz

    # 1) doesnt enter lisa waveband today. so also nt in the past (maybe near future)
    # 2) are currently in lisa band. maybe also in the past (but fro which point)
    # 3) are merged now. but they ahve been in lisa band in the past (and from which point)
    # 4) for both 2 and 3, we should filter out the 'interacting' systems

    ##############
    # determine (un)merged systems
    local_indices_merged_systems = local_indices[f_orb_now >= 1e2 * u.Hz]
    local_indices_unmerged_systems = local_indices[f_orb_now < 1e2 * u.Hz]
    config["logger"].warning(
        f"Of the total of {len(local_indices)} systems {len(local_indices_merged_systems)} are merged by today and {len(local_indices_unmerged_systems)} are not"
    )

    f_orb_now_unmerged_systems = f_orb_now[f_orb_now < 1e2 * u.Hz]

    ##############
    # determine unmerged systems in LISA passband
    query_unmerged_systems_within_LISA_passband = (
        f_orb_now_unmerged_systems >= lower_bound_LISA_passband
    ) & (f_orb_now_unmerged_systems <= upper_bound_LISA_passband)

    #
    local_indices_unmerged_systems_within_LISA_passband = (
        local_indices_unmerged_systems[query_unmerged_systems_within_LISA_passband]
    )

    print(
        f"Of the {len(local_indices_unmerged_systems)} unmerged systems {len(local_indices_unmerged_systems_within_LISA_passband)} are within the lisa frequency passband ([{lower_bound_LISA_passband},{upper_bound_LISA_passband}])"
    )

    # return only data from now unmerged systems within the lisa passband

    result_dict = select_dict_entries_with_new_indices(
        sampled_data_dict=result_dict,
        new_indices=local_indices_unmerged_systems_within_LISA_passband,
    )

    # add dists
    result_dict["dists"] = (
        dist[local_indices_unmerged_systems_within_LISA_passband] * u.kpc
    )

    print(result_dict)
    quit()
    return result_dict


LIGHTWEIGHT = False

###################
# Read T0 output
start = time.time()

if LIGHTWEIGHT:
    BinCodex_events_filename = (
        "/home/david/Desktop/bincodex_results/example_BinCodex.h5"
    )
else:
    BinCodex_events_filename = "/home/david/Desktop/bincodex_results/Seba_BinCodex.h5"

#
BinCodex_T0_events = pd.read_hdf(
    BinCodex_events_filename,
    "T0",
)

##################
# update T0 output

# get mass normalisation
mass_normalisation_fiducial = get_mass_norm(IC_model="fiducial", binary_fraction=0.5)

# set normalised yield
BinCodex_T0_events["normalized_yield"] = 1 / mass_normalisation_fiducial

# Query the dataset to select the formation of the WDs

# to check if things start with some number its easier to turn them into strings
BinCodex_T0_events["str_event"] = BinCodex_T0_events["event"].astype(str)
BinCodex_T0_events["str_type1"] = BinCodex_T0_events["type1"].astype(str)
BinCodex_T0_events["str_type2"] = BinCodex_T0_events["type2"].astype(str)

# first, lets query the type-changing events. Any type-change will do
wd_binaries = BinCodex_T0_events.query("str_event.str.startswith('1')")

# The type should change to a WD-type (and the other should already be one)
wd_binaries = wd_binaries.query("str_type1.str.startswith('2')")
wd_binaries = wd_binaries.query("str_type2.str.startswith('2')")

# to be sure lets only select the first ones that remain for each system
# Drop duplicates based on 'system_id', keeping only the first occurrence
# wd_binaries = wd_binaries.drop_duplicates(subset='UID', keep='first')

# lets delete the string versions of the columns again
wd_binaries = wd_binaries.drop(columns=["str_event", "str_type1", "str_type2"])

# lets also delete the original dataframe
del BinCodex_T0_events

stop = time.time()
print("created queried dataframe")
print("took {}".format(stop - start))

##################
#

# create file
input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# add group for events
input_hdf5_file.create_group("input_data/events")

# Write population config to file
input_hdf5_file.create_dataset("config/population", data=json.dumps({}))

# close
input_hdf5_file.close()

# store the data frame in the hdf5file
wd_binaries.to_hdf(input_hdf5_filename, key="input_data/events/stochastic_example")

#
convolution_config = copy.copy(default_convolution_config)
convolution_config["input_filename"] = input_hdf5_filename
convolution_config["output_filename"] = output_hdf5_filename
convolution_config["tmp_dir"] = TMP_DIR
convolution_config["redshift_interpolator_data_output_filename"] = os.path.join(
    TMP_DIR, "interpolator_dict.p"
)
convolution_config["multiply_by_time_binsize"] = False

###
# convolution instructions
convolution_config["convolution_instructions"] = [
    {
        "input_data_type": "event",
        "convolution_type": "sample",
        "input_data_name": "stochastic_example",
        "output_data_name": "stochastic_example",
        "ignore_metallicity": True,
        "post_convolution_function": post_convolution_function,
        "data_column_dict": {
            # required
            "normalized_yield": "normalized_yield",
            "delay_time": {"column_name": "time", "unit": u.Myr},
        },
    },
]

#
convolution_config["time_type"] = "lookback_time"
# convolution_config["convolution_lookback_time_bin_edges"] = np.arange(0, 4, 0.5) * u.Gyr

# construct the sfr-dict (NOTE: this uses absolute SFR, not metallicity dependent)
sfr_dict = {}
if LIGHTWEIGHT:
    sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 1) * u.Gyr).to(u.yr)
else:
    sfr_dict["lookback_time_bin_edges"] = (np.arange(0, 10, 0.25) * u.Gyr).to(u.yr)

sfr_dict["starformation_rate_array"] = (
    1e-6 * np.ones(sfr_dict["lookback_time_bin_edges"].shape[0] - 1) * u.Msun / u.yr
)  # example of a constant star-formation rate. this could be anything of course.

# store
convolution_config["SFR_info"] = sfr_dict

input_hdf5_file = h5py.File(input_hdf5_filename, "r")

# convolve
convolve(config=convolution_config)

print("finished convolution")


# read out content and integrate until today
with h5py.File(convolution_config["output_filename"], "r") as output_hdf5_file:
    print(
        output_hdf5_file[
            "output_data/event/stochastic_example/stochastic_example/convolution_results"
        ].keys()
    )

    formation_time_bin_keys = list(
        output_hdf5_file[
            "output_data/event/stochastic_example/stochastic_example/convolution_results"
        ].keys()
    )

    ################
    #
    total_in_waveband_lisa = 0

    # loop over the formation-time bins
    formation_time_bin_keys = sorted(
        formation_time_bin_keys, key=lambda x: float(x.split(" ")[0])
    )
    for formation_time_bin_key in formation_time_bin_keys:

        # formation_time_bin_key = "3500000000.0 yr"
        print("=================================")
        print(f"formation_time_bin_key: {formation_time_bin_key}")
        print("=================================")

        ###########
        # Read out data

        # convert units
        unit_dict = json.loads(
            output_hdf5_file[
                f"output_data/event/stochastic_example/stochastic_example/convolution_results/{formation_time_bin_key}"
            ].attrs["units"]
        )
        unit_dict = {key: u.Unit(val) for key, val in unit_dict.items()}
        print(unit_dict)

This example can be made more sophisticated by e.g.:
- Using a spatially-defined star-formation rate history. One can provide a list of starformation histories to the code, each element then representing a part of the grid where the SFR is defined in.
- Splitting 

## Orbit integration of stellar systems in a galaxy (including kicks!) using Agama

Several ingredients are necessary here: 

- population-synthesis results that contains binary systems that experience a supernova kick (or 2!)
- a galaxy star formation rate history model
- a galaxy potential and a method to place assign position and velocity to the system


One thing that is not taken into account is the torques on the binary system due to the background potential of the galaxy (nor any dynamical interactions, but lets go with the collissionless assumption of galaxies for now). These torques would alter their evolution and can not really be added post-processing. To be fully correct would need to be taken into account during the simulation of the binary system, but that means we need to evolve these systems `on-the-fly`, during the convolution, instead of using pre-simulated sets of population-synthesis results.


